*XGBOOST CON DATASET COMPLETO Y BALANCE PONDERADO*

In [ ]:
#XGBOOST CARGA ACADEMICA BALANCE PONDERADO
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

df_Comp_BP = pd.read_csv("/content/dataset1_completo_pesos.csv")

df_1 = df_Comp_BP.drop(columns=['mental_health_index', 'dropout_risk'])

X = df_1.drop(columns=["burnout_score", "peso"])
y = df_1["burnout_score"]
pesos = df_1["peso"]

In [ ]:
#DIVISIÓN DEL CONJUNTO DE DATOS
X_encoded = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test, pesos_train, pesos_test = train_test_split(
    X_encoded, y, pesos, test_size=0.30, random_state=42, stratify=y)

#PROFUNDIDADES A PROBAR
profundidades = [2, 3, 4, 5, 6, 7, 8, 9, 10]

#VARIABLES iNICIALIZADAS
mejor_accuracy_test = 0
mejor_depth = None
mejor_modelo = None
print("=== Iniciando Experimentos Variando la Profundidad (max_depth) ===")

for depth in profundidades:
    # CONFIGURACIÓN DEL MODELO XGBOOST PARA ESTA ITERACIÓN
    xgb_model_balanced = xgb.XGBClassifier(
        n_estimators=100,
        learning_rate=0.3,
        max_depth=depth,
        objective='binary:logistic',
        random_state=42)

    # ENTRENAMIENTO
    xgb_model_balanced.fit(X_train, y_train, sample_weight=pesos_train)

    # PREDICCIONES
    y_pred_train = xgb_model_balanced.predict(X_train)
    y_pred_test = xgb_model_balanced.predict(X_test)

    # CALCULAR EXACTITUD (ACCURACY) PARA ESTE PASO
    accuracy_train = accuracy_score(y_train, y_pred_train, sample_weight=pesos_train)
    accuracy_test = accuracy_score(y_test, y_pred_test, sample_weight=pesos_test)

    # Mostrar solo la exactitud de la iteración actual
    print(f"Profundidad: {depth} | Exactitud TRAIN: {accuracy_train:.4f} | Exactitud TEST: {accuracy_test:.4f}")

    # EVALUAR SI ESTE MODELO TIENE LA MEJOR EXACTITUD EN TEST HASTA AHORA
    if accuracy_test > mejor_accuracy_test:
        mejor_accuracy_test = accuracy_test
        mejor_depth = depth
        mejor_modelo = xgb_model_balanced

print(f"\n MEJOR MODELO ENCONTRADO (POR EXACTITUD) TIENE UNA PROFUNDIDAD DE: {mejor_depth}")

# Generar predicciones finales con el mejor modelo guardado
y_pred_train_mejor = mejor_modelo.predict(X_train)
y_pred_test_mejor = mejor_modelo.predict(X_test)

# Calcular todas las métricas para el mejor modelo por exactitud
# TRAIN
accuracy_train = accuracy_score(y_train, y_pred_train_mejor, sample_weight=pesos_train)
precision_train = precision_score(y_train, y_pred_train_mejor, sample_weight=pesos_train)
recall_train = recall_score(y_train, y_pred_train_mejor, sample_weight=pesos_train)
f1_train = f1_score(y_train, y_pred_train_mejor, sample_weight=pesos_train)

# TEST
accuracy_test = accuracy_score(y_test, y_pred_test_mejor, sample_weight=pesos_test)
precision_test = precision_score(y_test, y_pred_test_mejor, sample_weight=pesos_test)
recall_test = recall_score(y_test, y_pred_test_mejor, sample_weight=pesos_test)
f1_test = f1_score(y_test, y_pred_test_mejor, sample_weight=pesos_test)

# Proyectar todas las métricas juntas
print("\n Resultados del Modelo XGBOOST CON FACTORES CONDUCTUALES (MÉTRICAS DEL MEJOR MODELO)")
print(f"Métrica      |      TRAIN     |     TEST   ")
print(f"-------------|----------------|---------------")
print(f"Exactitud    |     {accuracy_train:.4f}     |    {accuracy_test:.4f}")
print(f"Precisión    |     {precision_train:.4f}     |    {precision_test:.4f}")
print(f"Sensibilidad |     {recall_train:.4f}     |    {recall_test:.4f}")
print(f"F1-Score     |     {f1_train:.4f}     |    {f1_test:.4f}")


=== Iniciando Experimentos Variando la Profundidad (max_depth) ===
Profundidad: 2 | Exactitud TRAIN: 0.8775 | Exactitud TEST: 0.8766
Profundidad: 3 | Exactitud TRAIN: 0.8785 | Exactitud TEST: 0.8761
Profundidad: 4 | Exactitud TRAIN: 0.8800 | Exactitud TEST: 0.8763
Profundidad: 5 | Exactitud TRAIN: 0.8833 | Exactitud TEST: 0.8756
Profundidad: 6 | Exactitud TRAIN: 0.8893 | Exactitud TEST: 0.8738
Profundidad: 7 | Exactitud TRAIN: 0.8994 | Exactitud TEST: 0.8725
Profundidad: 8 | Exactitud TRAIN: 0.9110 | Exactitud TEST: 0.8683
Profundidad: 9 | Exactitud TRAIN: 0.9278 | Exactitud TEST: 0.8645
Profundidad: 10 | Exactitud TRAIN: 0.9427 | Exactitud TEST: 0.8574

 MEJOR MODELO ENCONTRADO (POR EXACTITUD) TIENE UNA PROFUNDIDAD DE: 2

 Resultados del Modelo XGBOOST CON FACTORES CONDUCTUALES (MÉTRICAS DEL MEJOR MODELO)
Métrica      |      TRAIN     |     TEST   
-------------|----------------|---------------
Exactitud    |     0.8775     |    0.8766
Precisión    |     0.8613     |    0.8613
Sensibi

In [ ]:
#PRUEBA DE HIPERPARAMETROS
#BUSQUEDA OPTIMIZADA CON MAS VARIABLES QUE ESOLO EL DEPTH

X_encoded = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test, pesos_train, pesos_test = train_test_split(
    X_encoded, y, pesos, test_size=0.30, random_state=42, stratify=y)

# =============================================================================
# 2. CONFIGURACIÓN DEL ESPACIO DE BÚSQUEDA DE HIPERPARÁMETROS
# =============================================================================
param_grid = {
    'max_depth': [3, 4, 5, 6],               # Profundidades controladas
    'learning_rate': [0.01, 0.05, 0.1],      # Menos agresivo que 0.3
    'n_estimators': [200, 300, 500],         # Más árboles para compensar el learning_rate bajo
    'min_child_weight': [1, 3, 5],           # Restricción para evitar hojas muy específicas
    'subsample': [0.7, 0.8, 1.0],            # Muestreo aleatorio de filas
    'colsample_bytree': [0.7, 0.8, 1.0],     # Muestreo aleatorio de columnas
    'gamma': [0, 0.1, 0.5]                   # Penalización por crear nuevas divisiones
}

print("=== Iniciando Búsqueda Aleatoria de Hiperparámetros (RandomizedSearchCV) ===")

# Modelo base vacío
xgb_base = xgb.XGBClassifier(
    objective='binary:logistic',
    random_state=42
)

# Configuración del optimizador
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=20,           # Probará 20 combinaciones al azar (súbelo a 30 o 50 si tienes tiempo)
    scoring='accuracy',  # Métrica guía para elegir el mejor modelo
    cv=3,                # Validación cruzada de 3 pliegues
    random_state=42,
    n_jobs=-1            # Usa todos los núcleos de tu procesador para ir más rápido
)

# =============================================================================
# 3. ENTRENAMIENTO Y BÚSQUEDA
# =============================================================================
# Pasamos los pesos directamente al método fit
random_search.fit(X_train, y_train, sample_weight=pesos_train)

# Extraemos automáticamente el modelo ganador
mejor_modelo = random_search.best_estimator_

print(f"\nMEJORES HIPERPARÁMETROS ENCONTRADOS:\n{random_search.best_params_}\n")

# =============================================================================
# 4. PREDICCIONES FINALES Y CÁLCULO DE MÉTRICAS
# =============================================================================
y_pred_train_mejor = mejor_modelo.predict(X_train)
y_pred_test_mejor = mejor_modelo.predict(X_test)

# TRAIN
accuracy_train = accuracy_score(y_train, y_pred_train_mejor, sample_weight=pesos_train)
precision_train = precision_score(y_train, y_pred_train_mejor, sample_weight=pesos_train)
recall_train = recall_score(y_train, y_pred_train_mejor, sample_weight=pesos_train)
f1_train = f1_score(y_train, y_pred_train_mejor, sample_weight=pesos_train)

# TEST
accuracy_test = accuracy_score(y_test, y_pred_test_mejor, sample_weight=pesos_test)
precision_test = precision_score(y_test, y_pred_test_mejor, sample_weight=pesos_test)
recall_test = recall_score(y_test, y_pred_test_mejor, sample_weight=pesos_test)
f1_test = f1_score(y_test, y_pred_test_mejor, sample_weight=pesos_test)

# =============================================================================
# 5.RESULTADOS
# =============================================================================
print("Resultados del Modelo XGBOOST CON FACTORES CONDUCTUALES (MODELO OPTIMIZADO)")
print(f"Métrica      |      TRAIN     |     TEST   ")
print(f"-------------|----------------|---------------")
print(f"Exactitud    |     {accuracy_train:.4f}     |    {accuracy_test:.4f}")
print(f"Precisión    |     {precision_train:.4f}     |    {precision_test:.4f}")
print(f"Sensibilidad |     {recall_train:.4f}     |    {recall_test:.4f}")
print(f"F1-Score     |     {f1_train:.4f}     |    {f1_test:.4f}")

=== Iniciando Búsqueda Aleatoria de Hiperparámetros (RandomizedSearchCV) ===

MEJORES HIPERPARÁMETROS ENCONTRADOS:
{'subsample': 0.8, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 6, 'learning_rate': 0.1, 'gamma': 0, 'colsample_bytree': 0.8}

Resultados del Modelo XGBOOST CON FACTORES CONDUCTUALES (MODELO OPTIMIZADO)
Métrica      |      TRAIN     |     TEST   
-------------|----------------|---------------
Exactitud    |     0.8984     |    0.8744
Precisión    |     0.8749     |    0.8655
Sensibilidad |     0.9297     |    0.8865
F1-Score     |     0.9015     |    0.8759


*XGBOOST SIN FACTORES CONDUCTUALES BALANCE PONDERADO*

In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

df_3 = pd.read_csv("/content/dataset3_reducido_pesos.csv")
X = df_3.drop(columns=["burnout_score", "peso"])
y = df_3["burnout_score"]
pesos = df_3["peso"]
X_encoded = pd.get_dummies(X, drop_first=True)

In [ ]:
#DIVISIÓN DEL CONJUNTO DE DATOS
X_encoded = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test, pesos_train, pesos_test = train_test_split(
    X_encoded, y, pesos, test_size=0.30, random_state=42, stratify=y)

#PROFUNDIDADES A PROBAR
profundidades = [2, 3, 4, 5, 6, 7, 8, 9, 10]

#VARIABLES iNICIALIZADAS
mejor_accuracy_test = 0
mejor_depth = None
mejor_modelo = None
print("=== Iniciando Experimentos Variando la Profundidad (max_depth) ===")

for depth in profundidades:
    # CONFIGURACIÓN DEL MODELO XGBOOST PARA ESTA ITERACIÓN
    xgb_model_balanced = xgb.XGBClassifier(
        n_estimators=100,
        learning_rate=0.3,
        max_depth=depth,
        objective='binary:logistic',
        random_state=42)

    # ENTRENAMIENTO
    xgb_model_balanced.fit(X_train, y_train, sample_weight=pesos_train)

    # PREDICCIONES
    y_pred_train = xgb_model_balanced.predict(X_train)
    y_pred_test = xgb_model_balanced.predict(X_test)

    # CALCULAR EXACTITUD (ACCURACY) PARA ESTE PASO
    accuracy_train = accuracy_score(y_train, y_pred_train, sample_weight=pesos_train)
    accuracy_test = accuracy_score(y_test, y_pred_test, sample_weight=pesos_test)

    # Mostrar solo la exactitud de la iteración actual
    print(f"Profundidad: {depth} | Exactitud TRAIN: {accuracy_train:.4f} | Exactitud TEST: {accuracy_test:.4f}")

    # EVALUAR SI ESTE MODELO TIENE LA MEJOR EXACTITUD EN TEST HASTA AHORA
    if accuracy_test > mejor_accuracy_test:
        mejor_accuracy_test = accuracy_test
        mejor_depth = depth
        mejor_modelo = xgb_model_balanced

print(f"\n MEJOR MODELO ENCONTRADO (POR EXACTITUD) TIENE UNA PROFUNDIDAD DE: {mejor_depth}")

# Generar predicciones finales con el mejor modelo guardado
y_pred_train_mejor = mejor_modelo.predict(X_train)
y_pred_test_mejor = mejor_modelo.predict(X_test)

# Calcular todas las métricas para el mejor modelo por exactitud
# TRAIN
accuracy_train = accuracy_score(y_train, y_pred_train_mejor, sample_weight=pesos_train)
precision_train = precision_score(y_train, y_pred_train_mejor, sample_weight=pesos_train)
recall_train = recall_score(y_train, y_pred_train_mejor, sample_weight=pesos_train)
f1_train = f1_score(y_train, y_pred_train_mejor, sample_weight=pesos_train)

# TEST
accuracy_test = accuracy_score(y_test, y_pred_test_mejor, sample_weight=pesos_test)
precision_test = precision_score(y_test, y_pred_test_mejor, sample_weight=pesos_test)
recall_test = recall_score(y_test, y_pred_test_mejor, sample_weight=pesos_test)
f1_test = f1_score(y_test, y_pred_test_mejor, sample_weight=pesos_test)

# Proyectar todas las métricas juntas
print("\n Resultados del Modelo XGBOOST CON FACTORES CONDUCTUALES (MÉTRICAS DEL MEJOR MODELO)")
print(f"Métrica      |      TRAIN     |     TEST   ")
print(f"-------------|----------------|---------------")
print(f"Exactitud    |     {accuracy_train:.4f}     |    {accuracy_test:.4f}")
print(f"Precisión    |     {precision_train:.4f}     |    {precision_test:.4f}")
print(f"Sensibilidad |     {recall_train:.4f}     |    {recall_test:.4f}")
print(f"F1-Score     |     {f1_train:.4f}     |    {f1_test:.4f}")


=== Iniciando Experimentos Variando la Profundidad (max_depth) ===
Profundidad: 2 | Exactitud TRAIN: 0.8776 | Exactitud TEST: 0.8767
Profundidad: 3 | Exactitud TRAIN: 0.8782 | Exactitud TEST: 0.8762
Profundidad: 4 | Exactitud TRAIN: 0.8798 | Exactitud TEST: 0.8764
Profundidad: 5 | Exactitud TRAIN: 0.8830 | Exactitud TEST: 0.8758
Profundidad: 6 | Exactitud TRAIN: 0.8881 | Exactitud TEST: 0.8745
Profundidad: 7 | Exactitud TRAIN: 0.8960 | Exactitud TEST: 0.8726
Profundidad: 8 | Exactitud TRAIN: 0.9101 | Exactitud TEST: 0.8692
Profundidad: 9 | Exactitud TRAIN: 0.9239 | Exactitud TEST: 0.8663
Profundidad: 10 | Exactitud TRAIN: 0.9392 | Exactitud TEST: 0.8593

 MEJOR MODELO ENCONTRADO (POR EXACTITUD) TIENE UNA PROFUNDIDAD DE: 2

 Resultados del Modelo XGBOOST CON FACTORES CONDUCTUALES (MÉTRICAS DEL MEJOR MODELO)
Métrica      |      TRAIN     |     TEST   
-------------|----------------|---------------
Exactitud    |     0.8776     |    0.8767
Precisión    |     0.8613     |    0.8612
Sensibi

In [ ]:
#PRUEBA DE HIPERPARAMETROS
#BUSQUEDA OPTIMIZADA CON MAS VARIABLES QUE ESOLO EL DEPTH

X_encoded = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test, pesos_train, pesos_test = train_test_split(
    X_encoded, y, pesos, test_size=0.30, random_state=42, stratify=y)

# =============================================================================
# 2. CONFIGURACIÓN DEL ESPACIO DE BÚSQUEDA DE HIPERPARÁMETROS
# =============================================================================
param_grid = {
    'max_depth': [3, 4, 5, 6],               # Profundidades controladas
    'learning_rate': [0.01, 0.05, 0.1],      # Menos agresivo que 0.3
    'n_estimators': [200, 300, 500],         # Más árboles para compensar el learning_rate bajo
    'min_child_weight': [1, 3, 5],           # Restricción para evitar hojas muy específicas
    'subsample': [0.7, 0.8, 1.0],            # Muestreo aleatorio de filas
    'colsample_bytree': [0.7, 0.8, 1.0],     # Muestreo aleatorio de columnas
    'gamma': [0, 0.1, 0.5]                   # Penalización por crear nuevas divisiones
}

print("=== Iniciando Búsqueda Aleatoria de Hiperparámetros (RandomizedSearchCV) ===")

# Modelo base vacío
xgb_base = xgb.XGBClassifier(
    objective='binary:logistic',
    random_state=42
)

# Configuración del optimizador
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=20,           # Probará 20 combinaciones al azar (súbelo a 30 o 50 si tienes tiempo)
    scoring='accuracy',  # Métrica guía para elegir el mejor modelo
    cv=3,                # Validación cruzada de 3 pliegues
    random_state=42,
    n_jobs=-1            # Usa todos los núcleos de tu procesador para ir más rápido
)

# =============================================================================
# 3. ENTRENAMIENTO Y BÚSQUEDA
# =============================================================================
# Pasamos los pesos directamente al método fit
random_search.fit(X_train, y_train, sample_weight=pesos_train)

# Extraemos automáticamente el modelo ganador
mejor_modelo = random_search.best_estimator_

print(f"\nMEJORES HIPERPARÁMETROS ENCONTRADOS:\n{random_search.best_params_}\n")

# =============================================================================
# 4. PREDICCIONES FINALES Y CÁLCULO DE MÉTRICAS
# =============================================================================
y_pred_train_mejor = mejor_modelo.predict(X_train)
y_pred_test_mejor = mejor_modelo.predict(X_test)

# TRAIN
accuracy_train = accuracy_score(y_train, y_pred_train_mejor, sample_weight=pesos_train)
precision_train = precision_score(y_train, y_pred_train_mejor, sample_weight=pesos_train)
recall_train = recall_score(y_train, y_pred_train_mejor, sample_weight=pesos_train)
f1_train = f1_score(y_train, y_pred_train_mejor, sample_weight=pesos_train)

# TEST
accuracy_test = accuracy_score(y_test, y_pred_test_mejor, sample_weight=pesos_test)
precision_test = precision_score(y_test, y_pred_test_mejor, sample_weight=pesos_test)
recall_test = recall_score(y_test, y_pred_test_mejor, sample_weight=pesos_test)
f1_test = f1_score(y_test, y_pred_test_mejor, sample_weight=pesos_test)

# =============================================================================
# 5.RESULTADOS
# =============================================================================
print("Resultados del Modelo XGBOOST SIN FACTORES CONDUCTUALES (OPTIMIZADO)")
print(f"Métrica      |      TRAIN     |     TEST   ")
print(f"-------------|----------------|---------------")
print(f"Exactitud    |     {accuracy_train:.4f}     |    {accuracy_test:.4f}")
print(f"Precisión    |     {precision_train:.4f}     |    {precision_test:.4f}")
print(f"Sensibilidad |     {recall_train:.4f}     |    {recall_test:.4f}")
print(f"F1-Score     |     {f1_train:.4f}     |    {f1_test:.4f}")

=== Iniciando Búsqueda Aleatoria de Hiperparámetros (RandomizedSearchCV) ===

MEJORES HIPERPARÁMETROS ENCONTRADOS:
{'subsample': 0.8, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 6, 'learning_rate': 0.1, 'gamma': 0, 'colsample_bytree': 0.8}

Resultados del Modelo XGBOOST SIN FACTORES CONDUCTUALES (OPTIMIZADO)
Métrica      |      TRAIN     |     TEST   
-------------|----------------|---------------
Exactitud    |     0.8962     |    0.8749
Precisión    |     0.8724     |    0.8643
Sensibilidad |     0.9281     |    0.8895
F1-Score     |     0.8994     |    0.8767


*XGBOOST CON DATASET COMPLETO Y BALANCE SMOTE*

In [ ]:
#XGBOOST CARGA ACADEMICA BALANCE SMOTE
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

df_Comp_ST = pd.read_csv("/content/dataset1_completo_pesos.csv")

df_2 = df_Comp_ST.drop(columns=['mental_health_index', 'dropout_risk'])

X = df_2.drop(columns=["burnout_score", "peso"])
y = df_2["burnout_score"]




In [ ]:
from imblearn.over_sampling import SMOTE
X_encoded = pd.get_dummies(X, drop_first=True)

# Separación de los datos reales puros (TEST) antes de cualquier balanceo
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.30, random_state=42, stratify=y)

# =============================================================================
# BALANCEO DE CLASES SOLO EN EL CONJUNTO DE ENTRENAMIENTO (SMOTE)
# =============================================================================
smote = SMOTE(random_state=42)
X_train_balanceado, y_train_balanceado = smote.fit_resample(X_train, y_train)

# =============================================================================
# 2. CONFIGURACIÓN DEL ESPACIO DE BÚSQUEDA DE HIPERPARÁMETROS
# =============================================================================
param_grid = {
    'max_depth': [3, 4, 5, 6],               # Profundidades controladas
    'learning_rate': [0.01, 0.05, 0.1],      # Menos agresivo que 0.3
    'n_estimators': [200, 300, 500],         # Más árboles para compensar el learning_rate bajo
    'min_child_weight': [1, 3, 5],           # Restricción para evitar hojas muy específicas
    'subsample': [0.7, 0.8, 1.0],            # Muestreo aleatorio de filas
    'colsample_bytree': [0.7, 0.8, 1.0],     # Muestreo aleatorio de columnas
    'gamma': [0, 0.1, 0.5]                   # Penalización por crear nuevas divisiones
}

print("=== Iniciando Búsqueda Aleatoria de Hiperparámetros (RandomizedSearchCV) ===")

# Modelo base vacío
xgb_base = xgb.XGBClassifier(
    objective='binary:logistic',
    random_state=42
)

# Configuración del optimizador
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=20,           # Probará 20 combinaciones al azar (súbelo a 30 o 50 si tienes tiempo)
    scoring='accuracy',  # Métrica guía para elegir el mejor modelo
    cv=3,                # Validación cruzada de 3 pliegues
    random_state=42,
    n_jobs=-1            # Usa todos los núcleos de tu procesador para ir más rápido
)

# =============================================================================
# 3. ENTRENAMIENTO Y BÚSQUEDA
# =============================================================================
# Pasamos la data balanceada directamente al método fit
random_search.fit(X_train_balanceado, y_train_balanceado)

# Extraemos automáticamente el modelo ganador
mejor_modelo = random_search.best_estimator_

print(f"\nMEJORES HIPERPARÁMETROS ENCONTRADOS:\n{random_search.best_params_}\n")

# =============================================================================
# 4. PREDICCIONES FINALES Y CÁLCULO DE MÉTRICAS
# =============================================================================
# Evaluamos TRAIN contra la matriz balanceada
y_pred_train_mejor = mejor_modelo.predict(X_train_balanceado)

# Evaluamos TEST contra la matriz real y pura, sin SMOTE
y_pred_test_mejor = mejor_modelo.predict(X_test)

# TRAIN
accuracy_train = accuracy_score(y_train_balanceado, y_pred_train_mejor)
precision_train = precision_score(y_train_balanceado, y_pred_train_mejor)
recall_train = recall_score(y_train_balanceado, y_pred_train_mejor)
f1_train = f1_score(y_train_balanceado, y_pred_train_mejor)

# TEST
accuracy_test = accuracy_score(y_test, y_pred_test_mejor)
precision_test = precision_score(y_test, y_pred_test_mejor)
recall_test = recall_score(y_test, y_pred_test_mejor)
f1_test = f1_score(y_test, y_pred_test_mejor)

# =============================================================================
# 5.RESULTADOS
# =============================================================================
print("Resultados del Modelo XGBOOST CON FACTORES CONDUCTUALES (SMOTE)")
print(f"Métrica      |      TRAIN     |     TEST   ")
print(f"-------------|----------------|---------------")
print(f"Exactitud    |     {accuracy_train:.4f}     |    {accuracy_test:.4f}")
print(f"Precisión    |     {precision_train:.4f}     |    {precision_test:.4f}")
print(f"Sensibilidad |     {recall_train:.4f}     |    {recall_test:.4f}")
print(f"F1-Score     |     {f1_train:.4f}     |    {f1_test:.4f}")

=== Iniciando Búsqueda Aleatoria de Hiperparámetros (RandomizedSearchCV) ===

MEJORES HIPERPARÁMETROS ENCONTRADOS:
{'subsample': 0.8, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 6, 'learning_rate': 0.1, 'gamma': 0, 'colsample_bytree': 0.8}

Resultados del Modelo XGBOOST SIN FACTORES CONDUCTUALES (SMOTE)
Métrica      |      TRAIN     |     TEST   
-------------|----------------|---------------
Exactitud    |     0.9060     |    0.8708
Precisión    |     0.8845     |    0.4625
Sensibilidad |     0.9341     |    0.8537
F1-Score     |     0.9086     |    0.6000


*XGBOOST CON FEATURES REDUCIDOS Y BALANCE SMOTE*

In [ ]:

df_4 = pd.read_csv("/content/dataset3_reducido_pesos.csv")
X = df_4.drop(columns=["burnout_score", "peso"])
y = df_4["burnout_score"]
X_encoded = pd.get_dummies(X, drop_first=True)

In [ ]:
from imblearn.over_sampling import SMOTE
X_encoded = pd.get_dummies(X, drop_first=True)

# Separación de los datos reales puros (TEST) antes de cualquier balanceo
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.30, random_state=42, stratify=y)

# =============================================================================
# BALANCEO DE CLASES SOLO EN EL CONJUNTO DE ENTRENAMIENTO (SMOTE)
# =============================================================================
smote = SMOTE(random_state=42)
X_train_balanceado, y_train_balanceado = smote.fit_resample(X_train, y_train)

# =============================================================================
# 2. CONFIGURACIÓN DEL ESPACIO DE BÚSQUEDA DE HIPERPARÁMETROS
# =============================================================================
param_grid = {
    'max_depth': [3, 4, 5, 6],               # Profundidades controladas
    'learning_rate': [0.01, 0.05, 0.1],      # Menos agresivo que 0.3
    'n_estimators': [200, 300, 500],         # Más árboles para compensar el learning_rate bajo
    'min_child_weight': [1, 3, 5],           # Restricción para evitar hojas muy específicas
    'subsample': [0.7, 0.8, 1.0],            # Muestreo aleatorio de filas
    'colsample_bytree': [0.7, 0.8, 1.0],     # Muestreo aleatorio de columnas
    'gamma': [0, 0.1, 0.5]                   # Penalización por crear nuevas divisiones
}

print("=== Iniciando Búsqueda Aleatoria de Hiperparámetros (RandomizedSearchCV) ===")

# Modelo base vacío
xgb_base = xgb.XGBClassifier(
    objective='binary:logistic',
    random_state=42
)

# Configuración del optimizador
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=20,           # Probará 20 combinaciones al azar (súbelo a 30 o 50 si tienes tiempo)
    scoring='accuracy',  # Métrica guía para elegir el mejor modelo
    cv=3,                # Validación cruzada de 3 pliegues
    random_state=42,
    n_jobs=-1            # Usa todos los núcleos de tu procesador para ir más rápido
)

# =============================================================================
# 3. ENTRENAMIENTO Y BÚSQUEDA
# =============================================================================
# Pasamos la data balanceada directamente al método fit
random_search.fit(X_train_balanceado, y_train_balanceado)

# Extraemos automáticamente el modelo ganador
mejor_modelo = random_search.best_estimator_

print(f"\nMEJORES HIPERPARÁMETROS ENCONTRADOS:\n{random_search.best_params_}\n")

# =============================================================================
# 4. PREDICCIONES FINALES Y CÁLCULO DE MÉTRICAS
# =============================================================================
# Evaluamos TRAIN contra la matriz balanceada
y_pred_train_mejor = mejor_modelo.predict(X_train_balanceado)

# Evaluamos TEST contra la matriz real y pura, sin SMOTE
y_pred_test_mejor = mejor_modelo.predict(X_test)

# TRAIN
accuracy_train = accuracy_score(y_train_balanceado, y_pred_train_mejor)
precision_train = precision_score(y_train_balanceado, y_pred_train_mejor)
recall_train = recall_score(y_train_balanceado, y_pred_train_mejor)
f1_train = f1_score(y_train_balanceado, y_pred_train_mejor)

# TEST
accuracy_test = accuracy_score(y_test, y_pred_test_mejor)
precision_test = precision_score(y_test, y_pred_test_mejor)
recall_test = recall_score(y_test, y_pred_test_mejor)
f1_test = f1_score(y_test, y_pred_test_mejor)

# =============================================================================
# 5.RESULTADOS
# =============================================================================
print("Resultados del Modelo XGBOOST SIN FACTORES CONDUCTUALES (SMOTE)")
print(f"Métrica      |      TRAIN     |     TEST   ")
print(f"-------------|----------------|---------------")
print(f"Exactitud    |     {accuracy_train:.4f}     |    {accuracy_test:.4f}")
print(f"Precisión    |     {precision_train:.4f}     |    {precision_test:.4f}")
print(f"Sensibilidad |     {recall_train:.4f}     |    {recall_test:.4f}")
print(f"F1-Score     |     {f1_train:.4f}     |    {f1_test:.4f}")

=== Iniciando Búsqueda Aleatoria de Hiperparámetros (RandomizedSearchCV) ===

MEJORES HIPERPARÁMETROS ENCONTRADOS:
{'subsample': 0.8, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 6, 'learning_rate': 0.1, 'gamma': 0, 'colsample_bytree': 0.8}

Resultados del Modelo XGBOOST SIN FACTORES CONDUCTUALES (SMOTE)
Métrica      |      TRAIN     |     TEST   
-------------|----------------|---------------
Exactitud    |     0.8973     |    0.8641
Precisión    |     0.8745     |    0.4494
Sensibilidad |     0.9277     |    0.8783
F1-Score     |     0.9003     |    0.5946
